### 데이터 분석 컨설팅: 영도구청(신성장전략과)
* 인구통계지도 격자기준, 시간대(각시간대) 주거, 직장인구
* 인구통계지도 격자기준, 연령대 주거, 직장인구

In [25]:
import geopandas as gpd
import pandas as pd
import numpy as np
import glob
import PublicDataReader as pdr
import calendar

grid = gpd.read_file("grid/busan_50cell_v21.shp", encoding="utf-8")
grid['x'] = grid.centroid.x
grid['y'] = grid.centroid.y
ngprdc = gpd.read_file("26200/prdu_100m/nlsp_021001010.shp", encoding="utf-8")
ngprdc = ngprdc.to_crs(5179)

hcode = pdr.code_hdong()
hcode = hcode.loc[(hcode['시군구코드']=='26200') & (hcode['시군구명']!='') & (hcode['읍면동명']!='') & (hcode['말소일자']=='')]
hcode = hcode[['행정동코드', '읍면동명']]
hcode.columns = ['hcode', 'hname']

results1 = []
results2 = []

def time_pop(month, ym):
    time_lists = glob.glob(f"data/lifepop/*_time_p_{ym}.csv", recursive=True)
    for data in time_lists:
        df = pd.read_csv(data, sep="|", parse_dates=['std_ymd'], dtype={'hcode': str, 'id': str}, encoding="utf-8", low_memory=False)
        df = df.loc[df['hcode'].str.startswith('26200')].copy()
        df = pd.merge(df, hcode, "inner", "hcode")
        df = pd.merge(df, grid, "inner", "id")
        geom = gpd.points_from_xy(df.x, df.y, crs="epsg:5179")
        gdf = gpd.GeoDataFrame(df, geometry=geom)
        gdf = gpd.sjoin(gdf, ngprdc, "inner", predicate="within")
        cols_h = list(gdf.columns[3:27])
        cols_w = list(gdf.columns[27:51])
        cols_v = list(gdf.columns[51:75])
        gdf['std_ym'] = gdf['std_ymd'].astype(str).str[:7]
        results1.append(np.round((gdf.groupby(['std_ym', 'hname', 'gid'])[cols_h + cols_w + cols_v].sum() / 24 / calendar.monthrange(2023, int(month))[1]).reset_index().sort_values(by=['std_ym', 'gid'])))
        total1 = pd.concat(results1)
        total1 = total1.melt(id_vars=['std_ym', 'hname', 'gid'], value_vars=cols_h + cols_w + cols_v, var_name='class', value_name='pop')
        total1.loc[total1['class'].str.startswith('h'), 'regists'] = '주거'
        total1.loc[total1['class'].str.startswith('w'), 'regists'] = '직장'
        total1.loc[total1['class'].str.startswith('v'), 'regists'] = '방문'
        total1['times'] = total1['class'].str[4:]
        total1 = total1[['std_ym','hname','gid','regists','times','pop']]
        total1.columns = ['기준연월','행정동','gid','생활목적','시간대','인구수']
        total1.to_csv("26200/시간대_생활인구_100m.csv", sep="|", encoding="utf-8", index=False)

def ages_pop(month, ym):
    gender_lists = glob.glob(f"data/lifepop/*_sex_age_p_{ym}.csv", recursive=True)
    for data in gender_lists:
        df = pd.read_csv(data, sep="|", parse_dates=['std_ymd'], dtype={'hcode': str, 'id': str}, encoding="utf-8", low_memory=False)
        df = df.loc[df['hcode'].str.startswith('26200')].copy()
        df = pd.merge(df, hcode, "inner", "hcode")
        df = pd.merge(df, grid, "inner", "id")
        geom = gpd.points_from_xy(df.x, df.y, crs="epsg:5179")
        gdf = gpd.GeoDataFrame(df, geometry=geom)
        gdf = gpd.sjoin(gdf, ngprdc, "inner", predicate="within")
        gdf['h_10age'] = gdf.iloc[:, [4,5,18,19]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_10age'] = gdf.iloc[:, [32,33,46,47]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_10age'] = gdf.iloc[:, [60,61,74,75]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_20age'] = gdf.iloc[:, [6,7,20,21]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_20age'] = gdf.iloc[:, [34,35,48,49]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_20age'] = gdf.iloc[:, [62,63,76,77]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_30age'] = gdf.iloc[:, [8,9,22,23]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_30age'] = gdf.iloc[:, [36,37,50,51]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_30age'] = gdf.iloc[:, [64,65,78,79]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_40age'] = gdf.iloc[:, [10,11,24,25]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_40age'] = gdf.iloc[:, [38,39,52,53]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_40age'] = gdf.iloc[:, [66,67,80,81]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_50age'] = gdf.iloc[:, [12,13,26,27]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_50age'] = gdf.iloc[:, [40,41,54,55]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_50age'] = gdf.iloc[:, [68,69,82,83]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_60age'] = gdf.iloc[:, [14,15,28,29]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_60age'] = gdf.iloc[:, [42,43,56,57]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_60age'] = gdf.iloc[:, [70,71,84,85]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['h_70age'] = gdf.iloc[:, [16,30]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['w_70age'] = gdf.iloc[:, [44,58]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['v_70age'] = gdf.iloc[:, [72,86]].sum(axis=1) / calendar.monthrange(2023, int(month))[1]
        gdf['std_ym'] = gdf['std_ymd'].astype(str).str[:7]
        results2.append(np.round((gdf.groupby(['std_ym', 'hname', 'gid'])[['h_10age','w_10age','v_10age','h_20age','w_20age','v_20age',
                                                                          'h_30age','w_30age','v_30age','h_40age','w_40age','v_40age',
                                                                          'h_50age','w_50age','v_50age','h_60age','w_60age','v_60age',
                                                                          'h_70age','w_70age','v_70age'
                                                                         ]].sum()).reset_index().sort_values(by=['std_ym', 'gid'])))
        total2 = pd.concat(results2)
        total2 = total2.melt(id_vars=['std_ym', 'hname', 'gid'], value_vars=['h_10age','w_10age','v_10age','h_20age','w_20age','v_20age',
                                                                             'h_30age','w_30age','v_30age','h_40age','w_40age','v_40age',
                                                                             'h_50age','w_50age','v_50age','h_60age','w_60age','v_60age',
                                                                             'h_70age','w_70age','v_70age'],
                             var_name='class', value_name='pop')
        total2.loc[total2['class'].str.startswith('h'), 'regists'] = '주거'
        total2.loc[total2['class'].str.startswith('w'), 'regists'] = '직장'
        total2.loc[total2['class'].str.startswith('v'), 'regists'] = '방문'
        total2['ages'] = total2['class'].str[2:4]
        total2 = total2[['std_ym','hname','gid','regists','ages','pop']]
        total2.columns = ['기준연월','행정동','gid','생활목적','연령대','인구수']
        total2.to_csv("26200/연령대_생활인구_100m_01.csv", sep="|", encoding="utf-8", index=False)

if __name__ == "__main__":
    for month in range(1, 2):
        ym = f"2023{month:02d}"
        #time_pop(month, ym)
        ages_pop(month, ym)
